# 🌍 Carbon Outsourcing — Dashboard Generator

Run all cells top-to-bottom. The final cell downloads a standalone `carbon_outsourcing_dashboard.html` file.  
Open it in any browser — one **year slider** controls all five charts simultaneously.

**Charts included:**
- 🌍 World choropleth map (outsourced per capita)
- 📈 Production vs Consumption scatter (absolute + per capita)
- 📊 Top 15 importers bar chart
- 📊 Top 15 exporters bar chart

> **Data:** [Our World in Data — CO₂ Dataset](https://github.com/owid/co2-data)

---
## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import json, os, base64
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries ready.')

---
## Cell 2 — Load & Process Data

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'

AGGREGATES = [
    'World', 'Africa', 'Asia', 'Europe', 'North America', 'South America',
    'Oceania', 'European Union (27)', 'European Union (28)',
    'High-income countries', 'Low-income countries',
    'Lower-middle-income countries', 'Upper-middle-income countries',
    'OECD (Eora)', 'Non-OECD (Eora)', 'Asia (excl. China & India)',
    'Europe (excl. EU-27)', 'Europe (excl. EU-28)',
    'North America (excl. USA)', 'International transport', 'Antarctica',
]

print('⏳ Downloading data...')
raw = pd.read_csv(DATA_URL)

KEY = ['country', 'iso_code', 'year', 'co2', 'consumption_co2', 'population']
df = (
    raw[KEY]
    .query('country not in @AGGREGATES')
    .dropna(subset=['co2', 'consumption_co2', 'population', 'iso_code'])
    .query('population >= 1_000_000')
    .copy()
)

df['outsourced']            = df['consumption_co2'] - df['co2']
df['outsourced_per_capita'] = (df['outsourced'] / df['population']) * 1e6
df['co2_per_capita']        = (df['co2'] / df['population']) * 1e6
df['consumption_per_cap']   = (df['consumption_co2'] / df['population']) * 1e6
df['pop_millions']          = df['population'] / 1e6

years = sorted(df['year'].unique())
print(f'✅ {len(df):,} country-year rows | {len(years)} years: {years[0]}–{years[-1]}')

---
## Cell 3 — Generate & Download Dashboard

In [ ]:
# ── Build per-year JSON payload ────────────────────────────────────────────────
data_by_year = {}
for yr in years:
    ydf = df[df['year'] == yr]
    records = []
    for _, row in ydf.iterrows():
        records.append({
            'country':              row['country'],
            'iso_code':             row['iso_code'],
            'co2':                  round(float(row['co2']), 1),
            'consumption_co2':      round(float(row['consumption_co2']), 1),
            'outsourced':           round(float(row['outsourced']), 1),
            'outsourced_per_capita':round(float(row['outsourced_per_capita']), 2),
            'co2_per_capita':       round(float(row['co2_per_capita']), 2),
            'consumption_per_cap':  round(float(row['consumption_per_cap']), 2),
            'pop_millions':         round(float(row['pop_millions']), 1),
        })
    data_by_year[str(yr)] = records

years_list = [str(y) for y in years]
DATA_JSON  = json.dumps(data_by_year, separators=(',', ':'))
YEARS_JSON = json.dumps(years_list)
print(f'✅ Data payload: {len(DATA_JSON)/1024:.0f} KB across {len(years_list)} years')

# ── Decode the embedded HTML template ──────────────────────────────────────────
TEMPLATE_B64 = '...'
html_template = base64.b64decode(TEMPLATE_B64).decode('utf-8')

# ── Inject data ────────────────────────────────────────────────────────────────
html = html_template.replace('$$PLOTLY_DATA$$', DATA_JSON).replace('$$PLOTLY_YEARS$$', YEARS_JSON)

# ── Save ───────────────────────────────────────────────────────────────────────
out_file = 'carbon_outsourcing_dashboard.html'
with open(out_file, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'✅ Saved: {out_file}  ({len(html)/1024:.0f} KB)')

# ── Download in Colab ──────────────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(out_file)
    print('📦 Download started — check your browser downloads folder.')
except ImportError:
    print('(Not in Colab — open carbon_outsourcing_dashboard.html in your browser.)')